In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
import sys
from pathlib import Path

def find_repo_root(start: Path, marker=".git") -> Path:
    for parent in [start, *start.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find repo root from {start}")

repo_base = find_repo_root(Path.cwd())
# repo_base
tools_path = repo_base / "tools"
sys.path.append(str(tools_path))

In [5]:
from battle import *

def pull_by_id(num):
    return Battle(f'./../../data/replays/gen9-randombattle/gen9randombattle-{num}.json')

log_dir = repo_base / f"replays/gen9-randombattle/"
logs = Path('./../../data/replays/gen9-randombattle/')

In [13]:
bat = pull_by_id(2631886888)
bat.teams_full[1]['Florges']

{'name': 'Florges',
 'species': 'Florges-Blue',
 'speciesId': 'florges',
 'gender': 'F',
 'shiny': False,
 'level': 85,
 'moves': ['protect', 'moonblast', 'wish', 'calmmind'],
 'ability': 'Flower Veil',
 'evs': {'hp': 85, 'atk': 0, 'def': 85, 'spa': 85, 'spd': 85, 'spe': 85},
 'ivs': {'hp': 31, 'atk': 0, 'def': 31, 'spa': 31, 'spd': 31, 'spe': 31},
 'item': 'Leftovers',
 'teraType': 'Steel',
 'role': 'Bulky Setup',
 'bvs': {'hp': 78, 'atk': 65, 'def': 68, 'spa': 112, 'spd': 154, 'spe': 75},
 'stats': {'hp': 271,
  'atk': 115,
  'def': 164,
  'spa': 239,
  'spd': 311,
  'spe': 176}}

In [14]:
rows = []

stat_names = ["hp","atk","def","spa","spd","spe"]

for file in logs.iterdir():
    if (file.is_file() and (file.name[0]!='.')) :
        try : 
            battle = Battle(file)
        except UnicodeDecodeError as e : 
            print(f"Could not handle {file.name} : {e}")
            continue
        if not battle.custom_ruleQ:
            to_add = { # initial non-team data we may want to train on
                "id": battle.id.removeprefix("gen9randombattle-"),
                "rated": battle.rated,
                "duration": battle.end_time - battle.start_time,
                "p1": battle.p1.name,
                "p2": battle.p2.name,
                "p1_elo" : battle.p1.elo0,
                "p2_elo": battle.p2.elo0,
                "p1_wins" : battle.p1.name == battle.winner.name,
            }

            # team data and stats for each mon
            for i in range(2):
                for j, mon_name in enumerate(battle.teams_full[i].keys()):
                    poke = battle.teams_full[i][mon_name]
                    to_add[f"p{i+1}k{j+1}_id"] = poke["speciesId"]
                    for stat in stat_names:
                        to_add[f"p{i+1}k{j+1}_{stat}"] = poke["stats"][stat]
                    try:
                        for k in range(len(poke['types'])):
                            to_add[f"p{i+1}k{j+1}_T{k}"] = poke['types'][k]
                    except KeyError as e:
                        print(f"Error with {poke["speciesId"]} in {battle.id}")
                        continue
            rows.append(to_add)

In [7]:
matches = pd.DataFrame(rows)
matches.head(20)

,id,rated,duration,p1,p2,p1_elo,p2_elo,p1_wins,p1k1_id,p1k1_hp,...,p2k5_spa,p2k5_spd,p2k5_spe,p2k6_id,p2k6_hp,p2k6_atk,p2k6_def,p2k6_spa,p2k6_spd,p2k6_spe
0,2631906096,True,598,sufideu,saberclaw,1135,1140,False,quaquaval,264,...,141,321,141,klefki,233,139,201,183,194,174
1,2631763570,False,167,PineappleCats,L4V,1959,1949,False,greninja,246,...,166,149,174,toxtricity,257,165,162,234,162,170
2,2631369343,True,275,Chicken347,cococem,1999,2068,True,munkidori,269,...,185,207,207,electrodehisui,246,92,172,189,189,311
3,2631529004,False,123,WhatEver2102,Duck Cop,1999,1982,True,sinistcha,254,...,232,198,259,terapagos,265,105,175,145,175,137
4,2631993792,False,301,monomythic,OverthereStair,2120,2062,False,ambipom,263,...,146,146,100,trevenant,296,247,186,166,197,150
5,2631629401,False,116,Elite 4 Waally,jsjsiis,1958,1920,False,sudowoodo,284,...,112,244,173,probopass,257,105,316,188,325,125
6,2631479578,False,112,gamergray23,TheOldTimers,1928,1927,False,appletun,352,...,215,186,172,regice,284,93,226,226,402,138
7,2631439736,False,448,Mr Brightside,indias last hope,2115,2259,False,clodsire,343,...,209,209,209,reuniclus,337,119,182,270,200,103
8,2631771408,False,287,Illuminating_Fate,medo6037,2170,2177,True,sandyshocks,267,...,198,260,169,barraskewda,231,246,144,144,128,267
9,2631528750,True,326,Emily Montes,Elite 4 Waally,1907,1937,True,staraptor,263,...,137,137,113,cobalion,277,149,253,190,161,219


In [12]:
for i in range(2):
    for j in range(6):
        new_col_name = f"p{i+1}k{j+1}_atkM"
        old_col_names = [f"p{i+1}k{j+1}_atk", f"p{i+1}k{j+1}_spa"]
        matches[new_col_name] = matches[old_col_names].max(axis=1)
        matches = matches.drop(columns=old_col_names)